# Adding social and economic data

In [35]:
import pandas as pd


In [36]:
#transfers are denoted in thousands of rubles

transfers = pd.read_csv('../data/soc-econ/transfers.csv', sep=';')

transfers = transfers[transfers['object_level'] == "регион"]
transfers['share_of_total'] = transfers.groupby('year')['other_transfers'].transform(
    lambda x: 100 * (x / x.sum()) if x.sum() != 0 else 0)
transfers = transfers[transfers['year'] > 2018]
transfers = transfers[['object_name', 'year', 'other_transfers', 'share_of_total']]

transfers

,object_name,year,other_transfers,share_of_total
2,Амурская область,2024,1619943.4,0.161953
3,Еврейская автономная область,2024,3946294.6,0.394528
4,Забайкальский край,2024,19714297.1,1.970922
5,Камчатский край,2024,51388333.2,5.137510
6,Магаданская область,2024,5092353.6,0.509104
...,...,...,...,...
547,Волгоградская область,2019,20827300.9,2.647867
548,Краснодарский край,2019,22228524.8,2.826011
549,Республика Адыгея,2019,2813154.0,0.357649
550,Республика Калмыкия,2019,3194964.2,0.406190


In [37]:
#budgets are denoted in millions of rubles
budget = pd.read_csv('../data/soc-econ/budget.csv', sep=';')

budget = budget[budget["source"] == 'Доходы — всего']
budget = budget[budget["budget"] == 'Консолидированный бюджет субъекта']
budget = budget[budget['year'] > 2018]
budget = budget.groupby(['year', 'object_name'])['indicator_value'].sum().reset_index()
budget.columns = ['year', 'object_name', 'reg_inc']
budget = budget.merge(transfers, on = ['year', 'object_name'], how = 'left')
budget['other_transfers'] = budget['other_transfers']/1000
budget['share_of_income'] = budget['other_transfers']/budget['reg_inc']*100
budget

,year,object_name,reg_inc,other_transfers,share_of_total,share_of_income
0,2019,Алтайский край,126456.64,14280.1590,1.815500,11.292534
1,2019,Амурская область,83459.18,5009.3025,0.636855,6.002099
2,2019,Архангельская область,107888.59,12450.7983,1.582925,11.540422
3,2019,Астраханская область,59107.95,7686.2835,0.977191,13.003807
4,2019,Белгородская область,123221.52,7494.7127,0.952836,6.082308
...,...,...,...,...,...,...
493,2024,Чеченская Республика,111475.34,53495.0009,5.348123,47.988193
494,2024,Чувашская Республика,86445.26,20342.1602,2.033692,23.531840
495,2024,Чукотский автономный округ,43839.23,15630.7782,1.562675,35.654774
496,2024,Ямало-Ненецкий автономный округ,0.00,197.3367,0.019729,inf


In [38]:
#budgets are denoted in millions of rubles
budget = pd.read_csv('../data/soc-econ/budget.csv', sep=';')

budget = budget[budget["source"] == 'Доходы — всего']
budget = budget[budget['year'] > 2018]
budget = budget[budget["budget"] == 'Консолидированный бюджет субъекта']
budget = budget.groupby(['year', 'object_name'])['indicator_value'].sum().reset_index()
budget.columns = ['year', 'object_name', 'reg_inc']
budget = budget.merge(transfers, on = ['year', 'object_name'], how = 'left')
budget['other_transfers'] = budget['other_transfers']/1000
budget['share_of_income'] = budget['other_transfers']/budget['reg_inc']*100
budget

,year,object_name,reg_inc,other_transfers,share_of_total,share_of_income
0,2019,Алтайский край,126456.64,14280.1590,1.815500,11.292534
1,2019,Амурская область,83459.18,5009.3025,0.636855,6.002099
2,2019,Архангельская область,107888.59,12450.7983,1.582925,11.540422
3,2019,Астраханская область,59107.95,7686.2835,0.977191,13.003807
4,2019,Белгородская область,123221.52,7494.7127,0.952836,6.082308
...,...,...,...,...,...,...
493,2024,Чеченская Республика,111475.34,53495.0009,5.348123,47.988193
494,2024,Чувашская Республика,86445.26,20342.1602,2.033692,23.531840
495,2024,Чукотский автономный округ,43839.23,15630.7782,1.562675,35.654774
496,2024,Ямало-Ненецкий автономный округ,0.00,197.3367,0.019729,inf


In [39]:
health = pd.read_csv('../data/soc-econ/health.csv', sep=';')


selected_indicators = ['Численность лиц в возрасте 18 лет и старше, впервые признанных инвалидами, на 10 000 человек населения соответствующего возраста по субъектам Российской Федерации']
health = health[health["indicator_name"].isin(selected_indicators)]
health = health[health['year'] > 2018]
health = health[["object_name", "year", "indicator_value", "indicator_name"]]
health = health.pivot(index=["object_name", "year"], columns="indicator_name", values="indicator_value").reset_index()
health.rename(columns = {health.columns[2] : "disability_per10k"}, inplace=True)
health = health[["object_name", 'year', 'disability_per10k']]

health

indicator_name,object_name,year,disability_per10k
0,Алтайский край,2019,66.0
1,Алтайский край,2020,54.0
2,Алтайский край,2021,60.3
3,Алтайский край,2022,63.2
4,Амурская область,2019,46.9
...,...,...,...
379,Ямало-Ненецкий автономный округ,2022,28.1
380,Ярославская область,2019,45.8
381,Ярославская область,2020,46.8
382,Ярославская область,2021,41.6


In [40]:
pop = pd.read_csv('../data/soc-econ/population.csv', sep=';')

selected_indicators = ['Население субъектов Российской Федерации на 1 января',
                       'Городское население субъектов Российской Федерации на 1 января']

pop = pop[pop["indicator_name"].isin(selected_indicators)]
pop = pop[pop['year'] > 2018]
pop = pop[["object_name", "year", "indicator_value", "indicator_name"]]
pop = pop.pivot(index=["object_name", "year"], columns="indicator_name", values="indicator_value").reset_index()
pop["urban_share"] = pop["Городское население субъектов Российской Федерации на 1 января"]/pop["Население субъектов Российской Федерации на 1 января"]
pop["population"] = pop["Население субъектов Российской Федерации на 1 января"]*1000
pop = pop[["object_name", 'year', 'population', 'urban_share']]
pop

indicator_name,object_name,year,population,urban_share
0,Алтайский край,2019,2250100.0,0.572241
1,Алтайский край,2020,2224100.0,0.575604
2,Алтайский край,2021,2193000.0,0.578796
3,Алтайский край,2022,2154900.0,0.582394
4,Алтайский край,2023,2130900.0,0.583181
...,...,...,...,...
475,Ярославская область,2019,1243800.0,0.814842
476,Ярославская область,2020,1235600.0,0.814098
477,Ярославская область,2021,1221700.0,0.812474
478,Ярославская область,2022,1205600.0,0.810717


In [41]:
emp = pd.read_excel('../data/soc-econ/emp.xlsx')
selected_indicators = ['Уровень безработицы: Уровень безработицы']
emp = emp[emp["indicator_name"].isin(selected_indicators)]
emp = emp[emp["year"] > 2018]
emp = emp[["object_name", "year","indicator_value", "indicator_name"]]
emp = emp.pivot(index=["object_name", "year"], columns="indicator_name", values="indicator_value").reset_index()
emp.rename(columns = {emp.columns[2] : "uneployment"}, inplace=True)
emp

indicator_name,object_name,year,uneployment
0,Алтайский край,2019,5.8
1,Алтайский край,2020,5.9
2,Алтайский край,2021,5.5
3,Алтайский край,2022,3.7
4,Амурская область,2019,5.4
...,...,...,...
379,Ямало-Ненецкий автономный округ,2022,1.7
380,Ярославская область,2019,5.4
381,Ярославская область,2020,7.3
382,Ярославская область,2021,5.9


In [42]:
inc = pd.read_excel('../data/soc-econ/inc.xlsx')
selected_indicators = ['Численность населения с денежными доходами ниже границы бедности/величины прожиточного минимума',
                       'Медианный среднедушевой денежный доход населения', 'Потребительские расходы в среднем на душу населения',
                       'Среднедушевые денежные доходы населения']
inc = inc[inc["indicator_name"].isin(selected_indicators)]
inc = inc[inc["year"] > 2018]
inc = inc[["object_name", 'year', "indicator_value", "indicator_name"]]
inc = inc.pivot(index = ['object_name', 'year'], columns= 'indicator_name', values = 'indicator_value').reset_index()
inc.columns = ['object_name', 'year', 'median_income', 'consumption', 'av_income', "share_poverty"]
inc


,object_name,year,median_income,consumption,av_income,share_poverty
0,Алтайский край,2019,18932.4,18389.0,23993.0,17.6
1,Алтайский край,2020,19167.6,17726.0,23917.0,17.5
2,Алтайский край,2021,20783.2,20158.0,26010.0,16.5
3,Алтайский край,2022,24037.0,25062.0,31145.0,15.4
4,Амурская область,2019,25435.7,26407.0,33304.0,15.7
...,...,...,...,...,...,...
379,Ямало-Ненецкий автономный округ,2022,76957.8,47006.0,116639.0,4.5
380,Ярославская область,2019,23190.2,23029.0,28667.0,10.3
381,Ярославская область,2020,24094.4,22727.0,29527.0,9.9
382,Ярославская область,2021,26858.2,27391.0,33124.0,8.9


In [43]:
nat = pd.read_excel("../data/soc-econ/ethnic_composition.xlsx", sheet_name=None)
russian_percentages = {}
east_slav = {}
for region, data in nat.items():
    total_row = data[data.iloc[:, 0].str.contains("Указавшие национальную принадлежность", na=False)]
    total_population = total_row.iloc[0, 1] if not total_row.empty else None
    
    russian_row = data[data.iloc[:, 0].str.startswith("Русские", na=False)]
    belarus_row = data[data.iloc[:, 0].str.startswith("Белорусы", na=False)]
    ukraine_row = data[data.iloc[:, 0].str.startswith("Украинцы", na=False)]
    russian_population = russian_row.iloc[0, 1] if not russian_row.empty else None
    belarus_population = belarus_row.iloc[0, 1] if not belarus_row.empty else 0 
    ukraine_population = ukraine_row.iloc[0, 1] if not ukraine_row.empty else 0 

    if total_population and russian_population:
        percentage = (russian_population / total_population)*100
        east_slavs = ((russian_population + ukraine_population + belarus_population)/total_population)*100
        russian_percentages[region] = round(percentage, 2)
        east_slav[region] = round(east_slavs, 2)
    else:
        russian_percentages[region] = "NA"
        east_slav[region] = "NA"


df_rus = pd.DataFrame.from_dict(russian_percentages, orient='index', columns=['% Russians'])
df_es = pd.DataFrame.from_dict(east_slav, orient='index', columns=['% Slavs'])
df_rus.index.name = 'Region'
df_es.index.name = 'Region'
df_rus.reset_index(inplace=True)
df_es.reset_index(inplace=True)

df_rus = df_rus.merge(df_es, on = 'Region', how = 'left')
region_name_mapping = {
    'Архангельская область без автономного округа': 'Архангельская область без АО',
    'Ненецкий автономный округ': 'Ненецкий АО',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'г. Москва' : 'Москва',
    'г. Санкт-Петербург': 'Санкт-Петербург',
    'г. Севастополь': 'Севастополь',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Карачаево-Черкесская Республика': 'Карачаево-Черкесия',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Саха (Якутия)': 'Якутия',
    'РСО-Алания': 'Северная Осетия',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Тюменская область без автономных округов': 'Тюменская область без АО',
    'ХМАО' : 'Ханты-Мансийский АО',
    'ЯНАО' : 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
}

df_rus["Region"] = df_rus["Region"].replace(region_name_mapping)

In [51]:
df_se = budget.merge(pop, on = ['object_name', 'year'], how = 'left' )
df_se = df_se.merge(health, on = ['object_name', 'year'], how = 'left')
df_se = df_se.merge(emp, on = ['object_name', 'year'], how = 'left')
df_se = df_se.merge(inc, on = ['object_name', 'year'], how = 'left')


df_se.rename(columns = {df_se.columns[1] : "Region"}, inplace=True)

region_name_mapping = {
    'Архангельская область без автономного округа': 'Архангельская область без АО',
    'Ненецкий автономный округ': 'Ненецкий АО',
    'Еврейская автономная область':'Еврейская АО',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'Город Москва столица Российской Федерации город федерального значения': 'Москва',
    'Город Санкт-Петербург город федерального значения': 'Санкт-Петербург',
    'Город федерального значения Севастополь': 'Севастополь',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Карачаево-Черкесская Республика': 'Карачаево-Черкесия',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия — Алания': 'Северная Осетия',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Тюменская область без автономных округов': 'Тюменская область без АО',
    'Ханты-Мансийский автономный округ — Югра': 'Ханты-Мансийский АО',
    'Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
}

df_se["Region"] = df_se["Region"].replace(region_name_mapping)
df_se

,year,Region,reg_inc,other_transfers,share_of_total,share_of_income,population,urban_share,disability_per10k,uneployment,median_income,consumption,av_income,share_poverty
0,2019,Алтайский край,126456.64,14280.1590,1.815500,11.292534,2250100.0,0.572241,66.0,5.8,18932.4,18389.0,23993.0,17.6
1,2019,Амурская область,83459.18,5009.3025,0.636855,6.002099,786600.0,0.676710,46.9,5.4,25435.7,26407.0,33304.0,15.7
2,2019,Архангельская область,107888.59,12450.7983,1.582925,11.540422,1071600.0,0.771837,72.6,6.3,27990.9,29219.0,35724.0,13.6
3,2019,Астраханская область,59107.95,7686.2835,0.977191,13.003807,991100.0,0.653214,46.3,7.6,20134.4,21558.0,24971.0,15.5
4,2019,Белгородская область,123221.52,7494.7127,0.952836,6.082308,1550900.0,0.656844,53.5,3.9,25110.3,26119.0,32398.0,7.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,2024,Чеченская Республика,111475.34,53495.0009,5.348123,47.988193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
494,2024,Чувашская Республика,86445.26,20342.1602,2.033692,23.531840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
495,2024,Чукотский АО,43839.23,15630.7782,1.562675,35.654774,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
496,2024,Ямало-Hенецкий АО,0.00,197.3367,0.019729,inf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
df_full = df_se.merge(df_rus, on = 'Region', how = 'left')

df_full.to_csv("../data/intermediate/soc_econ_data.csv")